Krzysztof Jastrzębski 
nr indeksu: 149292

# Lab 1: Kafka w Pythonie — producent, konsument, ETL w czasie rzeczywistym
## Część 1: Część 1: Utwórz temat Kafka

In [1]:
! kafka-topics.sh --create --topic transactions --bootstrap-server broker:9092
! kafka-topics.sh --list --bootstrap-server broker:9092

Error while executing topic command : Topic 'transactions' already exists.
[2026-04-12 13:17:48,530] ERROR org.apache.kafka.common.errors.TopicExistsException: Topic 'transactions' already exists.
 (org.apache.kafka.tools.TopicCommand)
lab2
transactions


## Część 2: Producent — generowanie transakcji
### Zadanie 2.1

In [2]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


## Część 3: Konsument bezstanowy — filtrowanie
### Zadanie 3.1 — Filtruj duże transakcje

In [3]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id = 'filter',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# TWÓJ KOD
# Dla każdej wiadomości: sprawdź czy amount > 3000, jeśli tak — wypisz ALERT
# Format: ALERT: TX0042 | 2345.67 PLN | Warszawa | elektronika

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    transaction = message.value
    if(transaction['amount'] > 3000):
        print(f"ALERT: {transaction['tx_id']} | {transaction['amount']:.2f} PLN | {transaction['store']} | {transaction['category']}")
        

Overwriting consumer_filter.py


### Zadanie 3.2 — Transformacja i wzbogacenie

In [4]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

# TWÓJ KOD
# Czytaj z 'transactions' (użyj INNEGO group_id!)
consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id = 'enrichment',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

for message in consumer:
    transaction = message.value
    # Dodaj pole risk_level na podstawie amount
    if(transaction['amount'] > 3000):
        transaction['risk_level'] = 'HIGH'
    elif(transaction['amount'] > 1000):
        transaction['risk_level'] = 'MEDIUM'
    else:
        transaction['risk_level'] = 'LOW'
    # Wypisz wzbogaconą transakcję
    print(f"{transaction['tx_id']} | {transaction['amount']:.2f} PLN | {transaction['risk_level']} | {transaction['store']}")

Writing consumer_enrich.py


## Część 4: Konsument stanowy — agregacja
### Zadanie 4.1 - zliczanie transakcji przez sklep

In [5]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import defaultdict, Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id = 'counter',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = defaultdict(float)
msg_count = 0

# TWÓJ KOD
for message in consumer:
    msg_count += 1
    transaction = message.value
    # Dla każdej wiadomości:
    #   1. Zwiększ store_counts[store]
    store_counts[transaction['store']] += 1

    #   2. Dodaj amount do total_amount[store]
    total_amount[transaction['store']] += transaction['amount']
    #   3. Co 10 wiadomości wypisz tabelę:
    #      Sklep | Liczba | Suma | Średnia
    if msg_count % 10 == 0:
        print('Sklep | Liczba | Suma | Średnia')
        for store in store_counts:
             print(f"{store} | {store_counts[store]} | {total_amount[store]:.2f} PLN | {total_amount[store]/store_counts[store]:.2f} PLN")




Writing consumer_count.py


### Zadanie 4.2 — Statystyki per kategoria

In [6]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict, Counter
import json

# TWÓJ KOD
consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id = 'stats',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

msg_count = 0
category_count = Counter()
total_amount = defaultdict(float)
min_amount = defaultdict(lambda: float('inf'))
max_amount = defaultdict(lambda: float('-inf'))


for message in consumer:
    msg_count += 1
    transaction = message.value
    category_count[transaction['category']] += 1
    total_amount[transaction['category']] += transaction['amount']
    min_amount[transaction['category']] = min(min_amount[transaction['category']], transaction['amount'])
    max_amount[transaction['category']] = max(max_amount[transaction['category']], transaction['amount'])

    if msg_count % 10 == 0:
        print('Kategoria | Liczba transakcji | Suma transakcji | min | max')
        for category in category_count:
            print(f"{category} | {category_count[category]} | {total_amount[category]:.2f} PLN | {min_amount[category]:.2f} PLN | {max_amount[category]:.2f} PLN")
    
    
    


Writing consumer_stats.py


## Część 5: Wielu konsumentów
### Zadanie 5.2 — Pytania
1. **Co się stanie, jeśli uruchomisz consumer_filter.py po zakończeniu producenta?**
Consumer nie wyłapie żadnych wiadomości opublikowanych w topicu
2. **Co się stanie, jeśli dwóch konsumentów ma TĘ SAMĄ group_id?**
Kafka wysyła wiadomości tylko do jednego z konsumentów, drugi jest niewykorzystany (chyba, że topic zostanie podzielony na partycje - wtedy rozdziela load pomiędzy konsumentów).
3. **Jaka jest różnica między przetwarzaniem bezstanowym a stanowym?**
Przetwarzanie bezstanowe - każde zdarzenie przetwarzane w izolacji od innych
Przetwarzanie stanowe - zdarzenia przetwarzane z zachowaniem kontekstu innych zdarzeń

## Praca domowa

In [7]:
%%file velocity_anomalies.py
from kafka import KafkaConsumer
from collections import defaultdict
import json, time

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id = 'anomalies',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_60s_transactions = defaultdict(list)

for message in consumer:
    current_time = time.time()
    transaction = message.value
    user_60s_transactions[transaction['user_id']].append(current_time)
    user_60s_transactions[transaction['user_id']] = [t for t in user_60s_transactions[transaction['user_id']] if current_time - t <= 60]

    if(len(user_60s_transactions[transaction['user_id']]) > 3):
        print(f"ALERT: Użytkownik {transaction['user_id']} wykonał {len(user_60s_transactions[transaction['user_id']])} transakcji w ostatniej minucie")
    


Writing velocity_anomalies.py
